In [33]:
import pandas as pd
from datetime import datetime, time, timedelta, timezone
import joblib

In [34]:
price_data = pd.read_parquet('../../data/raw/xauusd_h1_2018_present.parquet')
df_raw = price_data.tail(300)
df_raw

,time,open,high,low,close,tick_volume,spread,real_volume
45300,2025-09-05 08:00:00,3558.50,3561.09,3555.79,3557.81,3567,7,0
45301,2025-09-05 09:00:00,3557.84,3558.07,3548.44,3552.63,4084,6,0
45302,2025-09-05 10:00:00,3552.69,3554.31,3546.70,3548.16,4256,5,0
45303,2025-09-05 11:00:00,3547.99,3549.83,3544.86,3546.04,3993,8,0
45304,2025-09-05 12:00:00,3546.04,3553.69,3546.01,3552.39,3811,5,0
...,...,...,...,...,...,...,...,...
45595,2025-09-24 04:00:00,3765.37,3771.85,3755.01,3756.58,4803,5,0
45596,2025-09-24 05:00:00,3756.49,3759.30,3750.74,3757.09,4149,12,0
45597,2025-09-24 06:00:00,3757.11,3766.59,3752.37,3765.45,3934,14,0
45598,2025-09-24 07:00:00,3765.42,3768.30,3763.24,3768.23,3156,6,0


In [35]:
def calculate_all_indicators(df):
    """Calculates a rich set of technical indicators on the DataFrame."""
    print("Calculating ~20 technical indicators...")
    
    # Use the pandas_ta Strategy builder for efficiency
    # We will list the indicators mentioned in the paper.
    # Note: Some names might be slightly different in pandas_ta.
    # 'kind' is the indicator name, you can find them in the pandas_ta documentation.
    my_study = ta.Study(
        name="RL Paper Indicators",
        description="A collection of ~20 indicators from the RL paper",
        ta=[
            # Momentum Indicators
            {"kind": "rsi"},          # Relative Strength Index
            {"kind": "mom"},          # Momentum
            {"kind": "stoch"},        # Stochastic Oscillator (%K and %D)
            {"kind": "macd"},         # Moving Average Convergence Divergence
            {"kind": "cci"},          # Commodity Channel Index
            {"kind": "roc"},          # Rate of Change
            {"kind": "cmo"},          # Chande Momentum Oscillator
            {"kind": "stochrsi"},     # Stochastic RSI
            {"kind": "willr"},        # Williams %R (similar to Ultimate Oscillator)
            
            # Trend Indicators
            {"kind": "adx"},          # Average Directional Movement Index
            {"kind": "trix"},         # TRIX
            {"kind": "psar"},         # Parabolic SAR
            {"kind": "tema"},         # Triple Exponential Moving Average
            {"kind": "trima"},        # Triangular Moving Average
            {"kind": "wma"},          # Weighted Moving Average
            {"kind": "dema"},         # Double Exponential Moving Average
            
            # Volume and Volatility Indicators
            {"kind": "mfi"},          # Money Flow Index
            {"kind": "bop"},          # Balance of Power
            {"kind": "atr"},          # Average True Range
        ]
    )
    
    # Run the strategy on the DataFrame (this appends all columns)
    df.ta.study(my_study)
    
    return df

In [36]:
df_raw['time'] = pd.to_datetime(df_raw['time'], unit='s')
df_raw.set_index('time', inplace=True)

# 2. Run the same feature engineering logic as our main script
# (This is a simplified, single-day version)
import pandas_ta as ta
df = df_raw.tz_localize('UTC')
df = calculate_all_indicators(df)

# 1. Specifically handle the PSAR columns by merging them.
# Find the exact column names by printing df.columns after calculating. They might have suffixes.
# Let's assume the columns are 'PSARl_0.02_0.2' and 'PSARs_0.02_0.2'
psar_long_col = 'PSARl_0.02_0.2'
psar_short_col = 'PSARs_0.02_0.2'

if psar_long_col in df.columns and psar_short_col in df.columns:
    # Create a single 'PSAR' column. Where 'PSARl' is NaN, it will use the value from 'PSARs'.
    df['PSAR'] = df[psar_long_col].fillna(df[psar_short_col])
    # Now we can drop the original two columns
    df.drop(columns=[psar_long_col, psar_short_col], inplace=True)
    print("Merged PSARl and PSARs into a single 'PSAR' column.")


# 3. Apply a general forward-fill for all remaining NaNs.
# This carries the last valid observation forward. It's the standard for time-series.
df.fillna(method='ffill', inplace=True)
print("Applied forward-fill (ffill) to remaining NaNs.")


# For demonstration, let's manually set 'today' to a date where we have data.
# The data ends on 2025-09-24. Let's pick a day from the last week of data.
today = datetime(2025, 9, 19).date()
# today = datetime.utcnow().date()
previous_day = today - timedelta(days=1)

asia_part1 = df.loc[str(previous_day)].between_time('22:00', '23:59')
asia_part2 = df.loc[str(today)].between_time('00:00', '07:59')
asia_session = pd.concat([asia_part1, asia_part2])

if asia_session.empty:
    print("Asian session data is empty. Cannot generate signal.")


asia_open = asia_session['open'].iloc[0]
asia_close = asia_session['close'].iloc[-1]
asia_high = asia_session['high'].max()
asia_low = asia_session['low'].min()
end_of_asia_ts = asia_session.index[-1]
    
features = {
    'day_of_week': today.weekday(),
    'asia_return': (asia_close - asia_open) / asia_open,
    'asia_range': asia_high - asia_low,
}

indicator_cols = [col for col in df.columns if col.startswith(('RSI', 'MOM', 'STOCH', 'MACD', 'CCI', 'ROC', 'CMO', 'WILLR', 'ADX', 'TRIX', 'PSAR', 'TEMA', 'TRIMA', 'WMA', 'DEMA', 'MFI', 'BOP', 'ATRr'))]

for col in indicator_cols:
    val = df.loc[end_of_asia_ts][col]
    features[col] = val
    
features_df = pd.DataFrame([features])

# 3. Load model and predict
# model = joblib.load("../../models/xgb_classifier_hyp_a_TUNED_paper_v2.joblib")
# prediction = model.predict(features_df)
# prediction

features_df


Calculating ~20 technical indicators...
Merged PSARl and PSARs into a single 'PSAR' column.
Applied forward-fill (ffill) to remaining NaNs.


C:\Users\mecha\AppData\Local\Temp\ipykernel_20804\2757107099.py:26: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)


,day_of_week,asia_return,asia_range,RSI_14,MOM_10,STOCHk_14_3_3,STOCHd_14_3_3,STOCHh_14_3_3,MACD_12_26_9,MACDh_12_26_9,...,TRIXs_30_9,PSARaf_0.02_0.2,PSARr_0.02_0.2,TEMA_10,TRIMA_10,WMA_10,DEMA_10,BOP,ATRr_14,PSAR
0,4,0.004817,27.93,55.276909,15.26,70.289427,50.194429,20.094998,-3.597413,2.084138,...,-0.010363,0.04,0.0,3653.18695,3643.190278,3647.289091,3649.139056,0.809221,9.654558,3628.3494


In [37]:
features_df.columns

Index(['day_of_week', 'asia_return', 'asia_range', 'RSI_14', 'MOM_10',
       'STOCHk_14_3_3', 'STOCHd_14_3_3', 'STOCHh_14_3_3', 'MACD_12_26_9',
       'MACDh_12_26_9', 'MACDs_12_26_9', 'CCI_14_0.015', 'ROC_10', 'CMO_14',
       'STOCHRSIk_14_14_3_3', 'STOCHRSId_14_14_3_3', 'WILLR_14', 'ADX_14',
       'ADXR_14_2', 'TRIX_30_9', 'TRIXs_30_9', 'PSARaf_0.02_0.2',
       'PSARr_0.02_0.2', 'TEMA_10', 'TRIMA_10', 'WMA_10', 'DEMA_10', 'BOP',
       'ATRr_14', 'PSAR'],
      dtype='object')